In [ ]:
%%writefile setup_voicevox.py
import os
import glob
import shutil
import subprocess
import time
import sys

def main():
    print("=======================================================")
    print("  🚀 BẮT ĐẦU QUY TRÌNH SETUP VOICEVOX NEMO ENGINE 🚀  ")
    print("=======================================================\n")

    # ---------------------------------------------------------
    # PHẦN 1: CHUẨN BỊ MÃ NGUỒN VÀ DỮ LIỆU
    # ---------------------------------------------------------
    print("🚀 1. Tải mã nguồn và cài đặt UV...")
    if not os.path.exists("/content/voicevox_nemo_engine"):
        os.system("git clone -q https://github.com/VOICEVOX/voicevox_nemo_engine.git /content/voicevox_nemo_engine")

    os.chdir("/content/voicevox_nemo_engine")
    os.system("curl -LsSf https://astral.sh/uv/install.sh | env UV_UNMANAGED_INSTALL='/usr/local/bin' sh")
    os.system("uv sync --quiet")

    print("🚀 2. Tải Core AI (GPU)...")
    os.system("apt-get install jq -y -q > /dev/null 2>&1")
    os.system("curl -s https://api.github.com/repos/VOICEVOX/voicevox_nemo_core/releases/latest | jq -r '.assets[] | select(.name | contains(\"linux\") and contains(\"gpu\") and contains(\"zip\")) | .browser_download_url' | xargs wget -qO core.zip")
    os.system("unzip -qo core.zip -d ./nemo_core")

    print("🚀 3. Tải dữ liệu nhân vật...")
    os.system("curl -s https://api.github.com/repos/VOICEVOX/voicevox_nemo_engine/releases/latest | jq -r '.assets[] | select(.name | contains(\"linux\") and contains(\"cpu\") and contains(\".vvpp\") and (contains(\".txt\") | not)) | .browser_download_url' | head -n 1 | xargs wget -qO engine_release.vvpp")
    os.system("unzip -qo engine_release.vvpp -d ./temp_fix")

    # Trích xuất thư mục nhân vật
    char_dirs = glob.glob("./temp_fix/**/resources/character_info", recursive=True)
    if char_dirs:
        if os.path.exists("./resources/character_info"):
            shutil.rmtree("./resources/character_info")
        shutil.copytree(char_dirs[0], "./resources/character_info")
        print("✅ Đã nạp thành công dữ liệu nhân vật!")

    os.system("rm -rf ./temp_fix engine_release.vvpp core.zip")
    print("🎉 Hoàn tất bước chuẩn bị dữ liệu!\n")

    # ---------------------------------------------------------
    # PHẦN 2: XỬ LÝ CUDA VÀ MÔI TRƯỜNG LÕI
    # ---------------------------------------------------------
    print("🧹 Dọn dẹp tiến trình cũ...")
    os.system("pkill -f 'run.py'")
    time.sleep(2)

    print("📦 4. Tải thư viện CUDA 11 & ép tải cuDNN bản 8...")
    install_cmd = 'pip install "nvidia-cudnn-cu11<9.0" nvidia-cufft-cu11 nvidia-cublas-cu11 nvidia-cuda-runtime-cu11 nvidia-cusparse-cu11 nvidia-curand-cu11'
    os.system(install_cmd)

    print("\n🔍 5. KIỂM TRA ĐỘ CHÍNH XÁC CỦA THƯ VIỆN cuDNN...")
    result = subprocess.run(["pip", "show", "nvidia-cudnn-cu11"], capture_output=True, text=True)

    if "WARNING" in result.stderr or "Name: nvidia-cudnn-cu11" not in result.stdout:
        print("❌ LỖI: Không tìm thấy gói nvidia-cudnn-cu11. Quá trình tải pip đã thất bại!")
        sys.exit(1)

    # Trích xuất Location một cách an toàn
    location = ""
    for line in result.stdout.split('\n'):
        if line.startswith("Location:"):
            location = line.split(":", 1)[1].strip()
            break

    if not location:
         print("❌ LỖI: Không tìm được đường dẫn Location.")
         sys.exit(1)

    print(f"   -> Thư mục cài đặt: {location}")

    check_cmd = f"find {location} -name 'libcudnn.so.8*'"
    found_files = subprocess.getoutput(check_cmd).strip()

    if not found_files or "No such file" in found_files:
        print("❌ LỖI CHÍ MẠNG: Thư mục tồn tại nhưng KHÔNG CÓ file libcudnn.so.8!")
        sys.exit(1)
    else:
        print(f"✅ BƯỚC KIỂM TRA HOÀN HẢO! Đã tìm thấy đúng cuDNN bản 8 tại:\n{found_files}\n")

    print("🔍 6. COPY VẬT LÝ THƯ VIỆN VÀO LÕI HỆ ĐIỀU HÀNH...")
    os.system(f"cp -L {location}/nvidia/*/lib/*.so* /usr/lib/x86_64-linux-gnu/ 2>/dev/null")
    os.system("ldconfig 2>/dev/null")

    # Kiểm tra xác nhận ở lõi Linux
    check_file = subprocess.getoutput("ls -l /usr/lib/x86_64-linux-gnu/libcudnn.so.8")
    if "No such file" in check_file:
        print("❌ LỖI: Vẫn không thấy cuDNN 8 sau khi copy.")
        sys.exit(1)
    else:
        print(f"✅ ĐÃ ÉP CÀI ĐẶT VÀO LÕI THÀNH CÔNG:\n{check_file}")

if __name__ == "__main__":
    main()

In [ ]:
!python setup_voicevox.py

In [ ]:
import os
import subprocess
import time
import glob

# Dọn dẹp tiến trình cũ...
os.system("pkill -f 'run.py'")
time.sleep(2)

print("🚀 Đang khởi động Server Voicevox ngầm...")
os.chdir("/content/voicevox_nemo_engine")
# Định vị thư mục Core AI
found_core = glob.glob("/content/voicevox_nemo_engine/nemo_core/**/libvoicevox_core.so", recursive=True)
real_core_dir = os.path.dirname(os.path.abspath(found_core[0])) if found_core else "/content/voicevox_nemo_engine/nemo_core"
log_file = open("engine.log", "w")
# Chạy Server và cắt đứt liên kết với Cell bằng start_new_session=True
process = subprocess.Popen(
    ["uv", "run", "run.py",
     "--use_gpu",
     "--host", "127.0.0.1",
     "--port", "50121",
     "--voicelib_dir", real_core_dir,
     "--runtime_dir", real_core_dir],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True  # <--- CHÌA KHÓA ĐỂ CELL KHÔNG BỊ TREO
)

print("⏳ Đang đợi Server khởi động ngầm (Sẽ không hiện log rác)...")
with open("engine.log", "r") as f:
    while True:
        line = f.readline()
        if line:
            # print(line.strip())  <--- ĐÃ THÊM DẤU # ĐỂ ẨN DÒNG NÀY ĐI
            # Nó vẫn đọc ngầm để tìm dòng báo thành công
            if "Uvicorn running on" in line or "Application startup complete" in line:
                print("-" * 50)
                print("\n🎉 THÀNH CÔNG TUYỆT ĐỐI! SERVER ĐÃ CHẠY XONG.")
                break
        else:
            if process.poll() is not None:
                print("\n❌ Server bị Crash. Hãy mở file engine.log ra để xem lỗi.")
                break
            time.sleep(0.5)

In [ ]:
!pip install aiohttp -q

import os
import wave
import io
import asyncio
import aiohttp
from IPython.display import Audio, display

base_url = "http://127.0.0.1:50121"
speaker_id = 10008
output_dir = "/content/voicevox_nemo_engine/tmp"
os.makedirs(output_dir, exist_ok=True)

# Đọc file
with open("/content/raw.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

# CÀI ĐẶT SỐ LƯỢNG REQUEST SONG SONG
# Không nên để quá cao (ví dụ 100) vì sẽ làm tràn RAM Colab hoặc dội bom làm sập Server.
# Mức 10 - 20 là mức tối ưu nhất cho GPU T4 trên Colab.
CONCURRENT_REQUESTS = 100
MAX_RETRIES = 3 # Số lần thử lại tối đa
semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

async def process_single_line(session, index, line):
    """Hàm xử lý cho 1 dòng độc lập, có cơ chế Retry khi thất bại"""
    async with semaphore:
        for attempt in range(MAX_RETRIES):
            try:
                # 1. Audio Query (Bất đồng bộ)
                async with session.post(f"{base_url}/audio_query", params={"text": line, "speaker": speaker_id}) as query_res:
                    if query_res.status != 200:
                        print(f"[{index}] ⚠️ Lỗi Query (Lần thử {attempt + 1}/{MAX_RETRIES}): {await query_res.text()}")
                        if attempt < MAX_RETRIES - 1:
                            await asyncio.sleep(1) # Nghỉ 1 giây trước khi thử lại
                            continue
                        return index, None # Hết lượt thử -> Chấp nhận thất bại
                        
                    query_data = await query_res.json()
                    
                # Chỉnh sửa tham số giọng nói
                query_data['speedScale'] = 1.05
                query_data['pitchScale'] = -0.05
                query_data['intonationScale'] = 1 # Cảm xúc
                query_data['volumeScale'] = 2
                # query_data["prePhonemeLength"] = 0.5 # nhịp nghỉ đầu
                # query_data["postPhonemeLength"] = 0.5 # nhịp nghỉ cuối
                query_data["outputSamplingRate"] = 48000

                # 2. Synthesis (Bất đồng bộ)
                async with session.post(f"{base_url}/synthesis", params={"speaker": speaker_id}, json=query_data) as synth_res:
                    if synth_res.status != 200:
                        print(f"[{index}] ⚠️ Lỗi Tổng hợp (Lần thử {attempt + 1}/{MAX_RETRIES}): {await synth_res.text()}")
                        if attempt < MAX_RETRIES - 1:
                            await asyncio.sleep(1)
                            continue
                        return index, None

                    audio_content = await synth_res.read()

                # Lưu file rời
                filename = os.path.join(output_dir, f"output_part_{index}.wav")
                with open(filename, "wb") as out_f:
                    out_f.write(audio_content)

                print(f"✅ Đã xong câu {index}/{len(lines)}")
                return index, audio_content # Thành công -> Trả về kết quả và thoát vòng lặp

            except Exception as e:
                print(f"[{index}] ⚠️ Lỗi kết nối (Lần thử {attempt + 1}/{MAX_RETRIES}): {e}")
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(1)
                    continue
                return index, None

async def main():
    print(f"🚀 BẮT ĐẦU XỬ LÝ SONG SONG {len(lines)} CÂU VỚI {CONCURRENT_REQUESTS} LUỒNG...")

    # Tạo phiên HTTP dùng chung để tăng tốc độ kết nối
    async with aiohttp.ClientSession() as session:
        # Tạo danh sách các tác vụ (tasks)
        tasks = [process_single_line(session, i + 1, line) for i, line in enumerate(lines)]

        # Chạy TẤT CẢ các tác vụ cùng một lúc và chờ đến khi tất cả xong
        results = await asyncio.gather(*tasks)

    print("\n🔄 Đang sắp xếp và ghép nối các đoạn âm thanh...")

    # RẤT QUAN TRỌNG: Do chạy song song, câu số 10 có thể xong trước câu số 2.
    # Ta phải sắp xếp lại (sort) kết quả dựa vào index để âm thanh không bị lộn xộn.
    results.sort(key=lambda x: x[0])

    audio_frames = []
    wav_params = None

    for index, audio_content in results:
        if audio_content:
            with wave.open(io.BytesIO(audio_content), 'rb') as w:
                if wav_params is None:
                    wav_params = w.getparams()
                audio_frames.append(w.readframes(w.getnframes()))

    # Xuất file cuối cùng
    if audio_frames:
        final_filename = os.path.join(output_dir, "output_final.wav")
        with wave.open(final_filename, 'wb') as w:
            w.setparams(wav_params)
            for frames in audio_frames:
                w.writeframes(frames)

        print(f"✨ HOÀN TẤT XUẤT FILE TỔNG: {final_filename}")
        display(Audio(final_filename))
    else:
        print("\n❌ Không có đoạn âm thanh nào được tạo thành công.")

# Kích hoạt chạy vòng lặp bất đồng bộ trong Colab
await main()